# 🔬 Experiment 2 - Part 3A: Delayed Injection Window (Later_33_48) [0-250 samples, 200tok]
**Mục tiêu:** Đánh giá can thiệp steering cửa sổ **Later_33_48** (`window_start=33`, `window_end=48`) với $\alpha_0=18.0$ ở độ dài `max_new_tokens=200` trên tập con **0-250** mẫu Test.
**Output file:** `exp02_part3A_later_33_48_200tok_0_250.json`
---

In [ ]:
# Cell 1: Cài đặt Dependencies
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm
print('✅ Dependencies installed successfully!')

In [ ]:
# Cell 2: Khởi tạo Environment, Seeds & Config
import os, json, glob, random, time, math, gc
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from bert_score import score as bert_score_eval
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
PART_NUM = 3
PART_NAME = 'Later_33_48'
SUB_PART = 'A'
START_IDX = 0
END_IDX = 250
ALPHA_0 = 18.0
WINDOW_START = 33
WINDOW_END = 48
MAX_NEW_TOKENS = 200
OUTPUT_JSON = 'exp02_part3A_later_33_48_200tok_0_250.json'

print(f'Config Loaded: Window=[{WINDOW_START}, {WINDOW_END}], Alpha={ALPHA_0}, Sample Range=[{START_IDX}, {END_IDX}], MaxTokens={MAX_NEW_TOKENS}')

In [ ]:
# Cell 3: Tải Dataset (Split 250 mẫu từ 500 Test Records)
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}', f'./{DATA_FILENAME}'
]
data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        data_path = matches[0]
        break
if not data_path:
    raise FileNotFoundError(f'❌ {DATA_FILENAME} not found!')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

shuffled_records = list(raw_dataset)
random.seed(SEED)
random.shuffle(shuffled_records)
n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
all_test = shuffled_records[n_train + n_val:]
test_500 = all_test[:500]
test_records = test_500[START_IDX:END_IDX]
print(f'📊 Loaded Sub-Split [{START_IDX}:{END_IDX}]: {len(test_records):,} records')

In [ ]:
# Cell 4: Load Qwen2.5 Model & Vector Steer
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f'Loading model {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('✅ Model loaded successfully!')

# Load or compute steering vector v_steer
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
if v_steer_paths:
    v_steer = torch.load(v_steer_paths[0], map_location='cpu')
    print('✅ Loaded v_steer.pt from Kaggle dataset input')
else:
    print('⚠️ v_steer.pt not found on input, generating synthetic normalized steering vector...')
    v_steer = torch.randn(model.config.hidden_size, dtype=torch.bfloat16)
    v_steer = v_steer / torch.norm(v_steer)


In [ ]:
# Cell 5: Dynamic Multi-GPU Device Safe Hook
class WindowedSteeringHook:
    def __init__(self, vector, alpha, start_step, end_step):
        self.vector = vector.detach().clone()
        self.alpha = alpha
        self.start_step = start_step
        self.end_step = end_step
        self.t = 0
        self.handle = None

    def hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            out_tensor = output[0]
        else:
            out_tensor = output

        # Prefill vs Decoding check
        if out_tensor.shape[1] > 1:
            self.t = 0
            return output

        self.t += 1
        if self.start_step <= self.t <= self.end_step:
            # Dynamic device & dtype casting for Multi-GPU compatibility
            steer_vec = self.vector.to(device=out_tensor.device, dtype=out_tensor.dtype)
            out_tensor[:, -1, :] = out_tensor[:, -1, :] + (self.alpha * steer_vec)

        if isinstance(output, tuple):
            return (out_tensor,) + output[1:]
        return out_tensor

    def register(self, layer_module):
        self.handle = layer_module.register_forward_hook(self.hook_fn)

    def remove(self):
        if self.handle:
            self.handle.remove()

print(f'Multi-GPU Safe WindowedSteeringHook prepared for [{WINDOW_START}, {WINDOW_END}] with alpha={ALPHA_0}')

In [ ]:
# Cell 6: Thực Thi Sinh 250 Câu với Hook Can Thiệp Cửa Sổ
hook = WindowedSteeringHook(v_steer, ALPHA_0, WINDOW_START, WINDOW_END)
target_layer = model.model.layers[8]
hook.register(target_layer)

generated_texts = []
eos_hits = 0
gen_lengths = []

print(f'🚀 Starting Generation for {len(test_records)} prompts [{START_IDX}:{END_IDX}]...')
start_time = time.time()

for item in tqdm(test_records):
    prompt = item['question']
    messages = [{'role': 'user', 'content': prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    generated_texts.append(gen_text)
    gen_lengths.append(len(gen_tokens))
    if tokenizer.eos_token_id in gen_tokens:
        eos_hits += 1

hook.remove()
total_time = time.time() - start_time
print(f'✅ Generation Completed in {total_time:.2f}s! EOS Hit Rate: {eos_hits/len(test_records)*100:.2f}%')

In [ ]:
# Cell 7: Lưu Trữ Raw Output để Merger Gộp Chỉnh Sửa Sau
raw_results = []
for idx, (rec, text, gen_len) in enumerate(zip(test_records, generated_texts, gen_lengths)):
    raw_results.append({
        "global_idx": START_IDX + idx,
        "question": rec['question'],
        "pos_ref": rec.get('pos_ref', rec.get('right_answer', '')),
        "neg_ref": rec.get('neg_ref', rec.get('hallucinated_answer', '')),
        "gen_text": text,
        "gen_length": gen_len
    })

output_data = {
    "experiment": f"Exp02_Part{PART_NUM}{SUB_PART}_{PART_NAME}_{MAX_NEW_TOKENS}tok",
    "window_name": PART_NAME,
    "sub_part": SUB_PART,
    "start_idx": START_IDX,
    "end_idx": END_IDX,
    "window_start": WINDOW_START,
    "window_end": WINDOW_END,
    "alpha_0": ALPHA_0,
    "max_new_tokens": MAX_NEW_TOKENS,
    "eos_hits": eos_hits,
    "total_samples": len(test_records),
    "records": raw_results
}

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)
print(f'✅ Sub-split raw results saved to {OUTPUT_JSON}')